# HCMAI Kaggle Inference Worker

**Chỉ cần thay giá trị `ROLE` ở Cell 2 rồi Run All.**

| ROLE | Kaggle Secret cần có | `MODELS` | Domain |
|---|---|---|---|
| `caption_ocr` | `caption_ocr_tunnel_token` | `caption,ocr` | `api.iamphuckhang.dev` |
| `asr_diarization` | `asr_diarization_token` | `asr,diarization` | `asr.iamphuckhang.dev` |
| `embedding` | `emb_tunnel_token` | `visual_emb,text_emb` | `emb.iamphuckhang.dev` |
| `preprocessing` | `preprocess_tunnel_token` | `transnet,gebd` | `preprocess.iamphuckhang.dev` |
| `dino` | `dino_tunnel_token` | `dino` | `dino.iamphuckhang.dev` |

> ⚠️ Enable **T4 GPU** + **Internet** trong Notebook Settings trước khi chạy.

In [ ]:
# Gán trực tiếp token vào biến
caption_ocr_tunnel_token = "eyJhIjoiZDY1ZDUwM2E1NWM3YWFhODZjYjI3OWU1NzEzMTkyOWMiLCJ0IjoiYjAyOGU0MmMtZjRjNi00NmQ1LWIzYjAtZjgzNzEzMjg3NTZlIiwicyI6IllUZzVNR1ZqT0dZdFpURTBZeTAwTTJJMExXRTRNR0l0TmpFd1pEQXlORFk0TURObCJ9" # Dán token thật của bạn vào đây
asr_diarization_token = "eyJhIjoiZDY1ZDUwM2E1NWM3YWFhODZjYjI3OWU1NzEzMTkyOWMiLCJ0IjoiMTE2YjM5YzUtNmU1MS00NGQwLTg1OTktYjdlZGE5N2VkZTM5IiwicyI6Ik16aG1OV1JoTTJFdFpEWTFNeTAwTVRjMUxXSm1ORE10TTJFM1lUbGpOVEV3WkdOaSJ9"
embedding_token = "eyJhIjoiZDY1ZDUwM2E1NWM3YWFhODZjYjI3OWU1NzEzMTkyOWMiLCJ0IjoiN2RhYmY3NWItZjZjNC00YmU1LTgyNjUtNmZmZjk4YmRkMzUwIiwicyI6IlpUaGhOV00xTkdRdE5UWmpaQzAwTmpVMkxUaGtNREl0T1daa056TTRNalV6TWpReCJ9"
preprocess_token = "eyJhIjoiZDY1ZDUwM2E1NWM3YWFhODZjYjI3OWU1NzEzMTkyOWMiLCJ0IjoiZjM1Nzk3ZDctNWNhMy00NjJiLWIxMDYtYmNlZmZlZjQzNDM0IiwicyI6Ik1HWTVOV1kxWXpZdFlqVTBPQzAwWTJOaUxXRTFabVV0TldFMU9UVmxORFUwTnpZMiJ9"
dino_token = "eyJhIjoiZDY1ZDUwM2E1NWM3YWFhODZjYjI3OWU1NzEzMTkyOWMiLCJ0IjoiZGJlNWFkOTEtYzEzYS00MDc3LWEyNjgtYWRiOWQ3ZjRlYjYxIiwicyI6Ik9EYzVZbU5qWXpJdE16VmpNaTAwT0RWa0xUZzJaV1l0TkRNME56Tm1ZMkZrTkRsaSJ9"

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  ĐỔI ROLE Ở ĐÂY — chỉ cần sửa dòng này thôi   ║
# ╚══════════════════════════════════════════════════╝
ROLE = "caption_ocr"

# ─── Mapping: role → (kaggle_secret_name, MODELS) ───────────────────────────
_ROLES = {
    "caption_ocr":     (caption_ocr_tunnel_token,  "caption,ocr"),
    "asr_diarization": (asr_diarization_token,      "asr,diarization"),
    "embedding":       (embedding_token,            "visual_emb,text_emb"),
    "preprocessing":   (preprocess_token,     "transnet,gebd"),
    "dino":            (dino_token,            "dino"),
}

if ROLE not in _ROLES:
    raise ValueError(f"Unknown ROLE={ROLE!r}. Valid: {list(_ROLES)}")

_token, _MODELS = _ROLES[ROLE]
print(f"Role      : {ROLE}")
# print(f"Secret    : {_SECRET_NAME}")
print(f"MODELS    : {_MODELS}")

In [ ]:
%%bash
# ── System deps: FFmpeg (needed by torchcodec inside sentence-transformers) ──
apt-get install -y --no-install-recommends ffmpeg libavutil-dev 2>&1 | tail -5

# ── cloudflared ──────────────────────────────────────────────────────────────
if ! [ -x ./cloudflared-linux-amd64 ]; then
    wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    chmod +x cloudflared-linux-amd64
    echo "cloudflared downloaded"
else
    echo "cloudflared already present"
fi

In [ ]:
import subprocess
from kaggle_secrets import UserSecretsClient

# Tokens are stored in Kaggle Secrets — never paste them into cells.
_token = UserSecretsClient().get_secret(_SECRET_NAME)

_tunnel = subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--no-autoupdate", "run", "--token", _token],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print(f"Tunnel started (PID {_tunnel.pid}) — role={ROLE}")

In [ ]:
%%bash
# Clone / update repo
if ! [ -d /kaggle/working/HCMAI_2026/.git ]; then
    git clone https://github.com/khang1108/MLeCDanBGold /kaggle/working/HCMAI_2026
else
    git -C /kaggle/working/HCMAI_2026 pull --ff-only
    echo "repo up to date"
fi

In [ ]:
%%bash
pip install -q -e '/kaggle/working/HCMAI_2026[preprocessing,embedding,transcripts]'

In [ ]:
import subprocess
if ROLE == 'preprocessing':
    print('Setting up TransNetV2 and EfficientGEBD models... (this might take a few minutes)')
    subprocess.run([
        'bash', '/kaggle/working/HCMAI_2026/src/hcmai/data/setup_models.sh'
    ], check=True)
    print('✅ Models setup complete.')

In [ ]:
import os

os.environ["MODELS"]                   = _MODELS
os.environ["HCMAI_PREPARATION_CONFIG"] = "/kaggle/working/HCMAI_2026/configs/preparation.s3.yaml"
os.environ["HCMAI_MODEL_CONFIG"]       = "/kaggle/working/HCMAI_2026/llm/config.yaml"
os.environ["HCMAI_ENRICHMENT_CONFIG"]  = "/kaggle/working/HCMAI_2026/configs/enrichment.yaml"

_server = subprocess.Popen(
    [
        "python", "-m", "uvicorn",
        "kaggle.inference_server:create_kaggle_app", "--factory",
        "--host", "127.0.0.1", "--port", "8100",
        "--log-level", "info",
    ],
    cwd="/kaggle/working/HCMAI_2026",
)
print(f"Server started (PID {_server.pid}) — MODELS={_MODELS}")

In [ ]:
import time, httpx

print("Waiting for models to load", end="", flush=True)
for _ in range(180):
    try:
        r = httpx.get("http://127.0.0.1:8100/health", timeout=2)
        if r.status_code == 200:
            break
    except Exception:
        pass
    time.sleep(1)
    print(".", end="", flush=True)

print()
ready = httpx.get("http://127.0.0.1:8100/ready", timeout=10).json()
print("ready =", ready.get("ready"))
for name, status in ready.get("models", {}).items():
    if status.get("enabled"):
        icon = "✅" if status.get("loaded") else "❌"
        print(f"  {icon} {name}: {status.get('checkpoint')}")

In [ ]:
print("Server and Tunnel are running! Keeping notebook alive...")
print("(If you are using Save Version, this cell will run forever so the session does not terminate.)")
try:
    _server.wait()
except KeyboardInterrupt:
    print("Stopped by user.")